In [6]:
import pandas as pd
import numpy as np
import math
from typing import List, Dict, Tuple

# ---------- 1. Data Loading & Preparation ----------
def load_data() -> pd.DataFrame:
    """Load job seekers data"""
    try:
        seekers_df = pd.read_csv('emplo.csv')[[
            'gender', 'birth_date', 'age', 'city', 'department',
            'sector', 'years_experience', 'highest_education',
            'contract_type', 'technical_skills', 'language_proficiency',
            'education_history', 'edu_value', 'salary'
        ]]
    except FileNotFoundError:
        # Create sample data if file not found
        seekers_df = pd.DataFrame([{
            'gender': 'Male', 'birth_date': '1990-05-15', 'age': 33,
            'city': 'Algiers', 'department': 'IT', 'sector': 'Technology',
            'years_experience': 5, 'highest_education': 'Master',
            'contract_type': 'Full-time', 'technical_skills': 'Python, SQL, Spark',
            'language_proficiency': 'English, French', 'education_history': 'University of Algiers',
            'edu_value': 3, 'salary': 70000
        }])
    
    return seekers_df

def input_job_details(seekers_df: pd.DataFrame) -> pd.DataFrame:
    """Collect job details from user input with validation"""
    valid_educations = seekers_df['highest_education'].unique().tolist()
    
    jobs = []
    job_counter = 1
    
    print("\nEnter Job Details (type 'done' when finished)")
    print("-------------------------------------------")
    
    while True:
        job = {}
        print(f"\nJob #{job_counter}")
        job_id = input(f"Job ID [default JOB-{job_counter:03}]: ").strip()
        job['job_id'] = job_id if job_id else f"JOB-{job_counter:03}"
        
        # Required Skills
        while True:
            skills = input("Required Skills (comma-separated): ").strip()
            if skills:
                job['required_skills'] = [s.strip() for s in skills.split(',')]
                break
            print("Error: At least one skill required!")

        # Education Validation
        while True:
            edu = input(f"Minimum Education (valid options: {', '.join(valid_educations)}): ").strip().title()
            if edu in valid_educations:
                job['min_education'] = edu
                break
            print(f"Error: Invalid education level. Please choose from {valid_educations}")

        # Experience Validation
        while True:
            exp = input("Minimum Experience (years): ").strip()
            if exp.isdigit() and int(exp) >= 0:
                job['min_experience'] = int(exp)
                break
            print("Error: Please enter a non-negative integer")

        # Salary Validation
        while True:
            salary = input("Salary Offer: ").strip()
            if salary.replace(',', '').isdigit():
                job['salary_offer'] = int(salary.replace(',', ''))
                break
            print("Error: Please enter a valid number")

        # Location and Sector
        job['location'] = input("Location: ").strip().title()
        job['sector'] = input("Sector: ").strip().title()

        # Contract Type Validation
        while True:
            contract = input("Contract Type (e.g., CDD, CDI, Freelance): ").strip().upper()
            if contract:
                job['contract_type'] = contract
                break
            print("Error: Contract type cannot be empty")

        jobs.append(job)
        job_counter += 1

        if input("\nAdd another job? (y/n): ").lower() != 'y':
            break

    return pd.DataFrame(jobs)

# ---------- 2. Core Matching Functions ----------
def preprocess_data(seekers_df: pd.DataFrame, jobs_df: pd.DataFrame) -> Tuple:
    """Preprocess and normalize all data for matching"""
    # Create education mapping from seekers data
    education_mapping = seekers_df.groupby('highest_education')['edu_value'].first().to_dict()
    
    # Set education ranks
    seekers_df['edu_rank'] = seekers_df['edu_value']
    jobs_df['edu_rank'] = jobs_df['min_education'].map(education_mapping).fillna(-1).astype(int)

    # Skills processing
    seekers_df['technical_skills'] = seekers_df['technical_skills'].fillna('').str.lower()
    seeker_skills = [set(s.split(', ')) for s in seekers_df['technical_skills']]
    job_skills = [set(map(str.lower, req)) for req in jobs_df['required_skills']]

    # Sector and contract type normalization
    all_sectors = pd.Categorical(seekers_df['sector'].tolist() + jobs_df['sector'].tolist())
    seek_sector_codes = pd.Categorical(seekers_df['sector'], categories=all_sectors.categories).codes
    job_sector_codes = pd.Categorical(jobs_df['sector'], categories=all_sectors.categories).codes

    contract_types = pd.Categorical(seekers_df['contract_type'].tolist() + jobs_df['contract_type'].tolist())
    seek_cont_codes = pd.Categorical(seekers_df['contract_type'], categories=contract_types.categories).codes
    job_cont_codes = pd.Categorical(jobs_df['contract_type'], categories=contract_types.categories).codes

    return (seekers_df, jobs_df, seeker_skills, job_skills, 
            seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes)

def calculate_features(seekers_df, jobs_df, seeker_skills, job_skills, 
                      seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes) -> np.ndarray:
    """Calculate all feature scores between seekers and jobs"""
    N_seek = len(seekers_df)
    N_job = len(jobs_df)
    F_raw = np.zeros((N_seek, N_job, 7), dtype=np.float32)

    # Skill TF-IDF calculation
    all_skills = [skill for skills in seekers_df['technical_skills'] for skill in skills.split(', ') if skill]
    skill_counts = pd.Series(all_skills).value_counts()
    total_docs = N_seek + N_job
    skill_idf = {s: np.log((total_docs + 1)/(cnt + 1)) + 1 for s, cnt in skill_counts.items()}

    # Salary normalization
    all_salaries = np.concatenate([seekers_df['salary'].values, jobs_df['salary_offer'].values])
    max_sal = np.percentile(all_salaries, 95)
    min_sal = np.percentile(all_salaries, 5)

    # Education normalization
    max_edu_value = seekers_df['edu_value'].max()

    for i in range(N_seek):
        for j in range(N_job):
            # Skill match (40%)
            common_skills = seeker_skills[i] & job_skills[j]
            missing_skills = job_skills[j] - seeker_skills[i]
            penalty = 1 - (len(missing_skills) / len(job_skills[j]))
            match_score = sum(skill_idf.get(s, 0) for s in common_skills)
            total_score = sum(skill_idf.get(s, 0) for s in job_skills[j])
            F_raw[i,j,0] = penalty * (match_score / total_score if total_score > 0 else 0)

            # Experience (15%)
            F_raw[i,j,1] = min(seekers_df.at[i,'years_experience'] / max(jobs_df.at[j,'min_experience'], 1), 1.5)

            # Salary (15%)
            salary_diff = abs(jobs_df.at[j,'salary_offer'] - seekers_df.at[i,'salary'])
            F_raw[i,j,2] = 1 - np.log1p(salary_diff) / np.log1p(max_sal - min_sal)

            # Education (10%)
            F_raw[i,j,3] = seekers_df.at[i,'edu_value'] / max_edu_value

            # Sector (10%)
            F_raw[i,j,4] = (seek_sector_codes[i] == job_sector_codes[j])

            # Contract (5%)
            F_raw[i,j,5] = (seek_cont_codes[i] == job_cont_codes[j])

            # Education history (5% - using original edu_value column)
            F_raw[i,j,6] = seekers_df.at[i,'edu_value'] / 20

    # Min-Max Normalization
    feature_mins = F_raw.min(axis=(0,1))
    feature_maxs = F_raw.max(axis=(0,1))
    F = (F_raw - feature_mins) / (feature_maxs - feature_mins + 1e-8)
    
    return F

# ---------- 3. Genetic Algorithm Implementation ----------
# ... (Keep all genetic algorithm functions identical to previous version)

# ---------- 4. Main Execution ----------
if __name__ == '__main__':
    # Load seekers data
    seekers_df = load_data()
    
    # Get jobs from user input
    print("=== Job Entry System ===")
    jobs_df = input_job_details(seekers_df)
    
    # Preprocess data
    (seekers_df, jobs_df, seeker_skills, job_skills, 
     seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes) = preprocess_data(seekers_df, jobs_df)
    
    # Calculate feature matrix
    F = calculate_features(seekers_df, jobs_df, seeker_skills, job_skills,
                          seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes)
    
    # Define weights for each feature
    WEIGHTS = np.array([0.40, 0.15, 0.15, 0.10, 0.10, 0.05, 0.05], dtype=np.float32)
    
    # Create valid jobs matrix
    valid_jobs = (
        (seekers_df['edu_rank'].values[:, None] >= jobs_df['edu_rank'].values[None, :]) &
        (seekers_df['years_experience'].values[:, None] >= jobs_df['min_experience'].values[None, :]) &
        np.array([
            [len(seeker_skills[i] & job_skills[j]) >= max(1, len(job_skills[j]) // 2) 
             for j in range(len(jobs_df))]
            for i in range(len(seekers_df))
        ])
    )
    
    # Run genetic algorithm
    best_solution, best_score = run_genetic_algorithm(F, WEIGHTS, valid_jobs)
    
    # Display results
    print("\n=== Matching Results ===")
    print("Best Solution Score:", best_score)
    print("\nOptimal Matches:")
    for i, job_idx in enumerate(best_solution):
        if job_idx >= 0:
            seeker = seekers_df.iloc[i]
            job = jobs_df.iloc[job_idx]
            
            common_skills = seeker_skills[i] & job_skills[job_idx]
            missing_skills = job_skills[job_idx] - seeker_skills[i]
            sector_match = seeker['sector'] == job['sector']
            contract_match = seeker['contract_type'] == job['contract_type']
            
            print(f"\nSeeker {i} → Job {job['job_id']}")
            print(f"  Skills: {len(common_skills)}/{len(job_skills[job_idx])} matched")
            if missing_skills:
                print(f"  Missing Skills: {', '.join(missing_skills)}")
            print(f"  Sector: {'Match' if sector_match else f'Mismatch (Seeker: {seeker["sector"]}, Job: {job["sector"]})'}")
            print(f"  Contract: {'Match' if contract_match else f'Mismatch (Seeker: {seeker["contract_type"]}, Job: {job["contract_type"]})'}")
            print(f"  Experience: {seeker['years_experience']}y (Req: {job['min_experience']}y)")
            print(f"  Education: {seeker['highest_education']} (Req: {job['min_education']})")
            print(f"  Salary: Seeker ${seeker['salary']:,} vs Job Offer ${job['salary_offer']:,}")
    
    # Show top candidates per job
    print("\n=== Top Candidates per Job ===")
    for j in range(len(jobs_df)):
        job = jobs_df.iloc[j]
        scores = F[:,j].dot(WEIGHTS)
        valid = valid_jobs[:,j]
        ranked = sorted([(i, scores[i]) for i in np.where(valid)[0]], key=lambda x: -x[1])[:5]
        
        print(f"\nJob {job['job_id']} ({job['sector']}):")
        for rank, (i, score) in enumerate(ranked, 1):
            seeker = seekers_df.iloc[i]
            common_skills = seeker_skills[i] & job_skills[j]
            print(f"{rank}. [Score: {score:.2%}] {seeker['technical_skills']}")
            print(f"   Sector: {seeker['sector']} | Exp: {seeker['years_experience']}y | Edu: {seeker['highest_education']}")
            print(f"   Matching Skills: {', '.join(common_skills)}")

=== Job Entry System ===

Enter Job Details (type 'done' when finished)
-------------------------------------------

Job #1
Generation 100/100 | Best Score: -800.52

=== Matching Results ===
Best Solution Score: -800.5234965443611

Optimal Matches:

Seeker 1179 → Job 1
  Skills: 1/1 matched
  Sector: Match
  Contract: Mismatch (Seeker: Freelance, Job: CDD)
  Experience: 12y (Req: 3y)
  Education: Ingénieur d'État (Req: Master)
  Salary: Seeker $54,200.0 vs Job Offer $60,000

Seeker 1245 → Job 1
  Skills: 1/1 matched
  Sector: Match
  Contract: Match
  Experience: 23y (Req: 3y)
  Education: Ingénieur d'État (Req: Master)
  Salary: Seeker $73,500.0 vs Job Offer $60,000

Seeker 4544 → Job 1
  Skills: 1/1 matched
  Sector: Match
  Contract: Match
  Experience: 21y (Req: 3y)
  Education: Doctorat (Req: Master)
  Salary: Seeker $70,000.0 vs Job Offer $60,000

Seeker 4564 → Job 1
  Skills: 1/1 matched
  Sector: Match
  Contract: Mismatch (Seeker: Stage, Job: CDD)
  Experience: 21y (Req: 3y)
 